# 🦅 Áquila Pro — A100 Sovereign Neural Engine
Este notebook executa o motor neural de alta performance em **GPU NVIDIA A100** para processamento de tarefas complexas do Áquila Pro.

### ⚡ Configuração Inicial de Runtime:
1. No menu superior: **Ambiente de execução (Runtime)** > **Alterar tipo de ambiente de execução**.
2. Selecione **GPU** e escolha **A100 GPU** com perfil **High-RAM**.

In [1]:
# 1. Verificação de Hardware NVIDIA A100 & CUDA
!nvidia-smi

import torch
import sys, os, time

print(f"\n[ÁQUILA] PyTorch: {torch.__version__}")
print(f"[ÁQUILA] CUDA Disponível: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[ÁQUILA] Dispositivo Detectado: {gpu_name}")
    print(f"[ÁQUILA] VRAM Total: {vram_gb:.2f} GB")
    print(f"[ÁQUILA] Suporte a BF16 (Tensor Cores Gen 3): {torch.cuda.is_bf16_supported()}")
    if "A100" in gpu_name:
        print("\n🔥 SUCESSO: GPU NVIDIA A100 ATIVADA E PRONTA PARA OPERAÇÕES PESADAS!")
    else:
        print(f"\n⚠️ Atenção: GPU atual é {gpu_name}. Para ativar A100, mude o tipo de runtime para A100.")
else:
    print("❌ Nenhuma GPU detectada. Habilite a GPU nas configurações de runtime.")

Wed Oct 21 15:49:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# 2. Download do Dataset Oficial (HuggingFace)
!pip install datasets pandas transformers -q

from datasets import load_dataset
import pandas as pd

print('[ÁQUILA] Estabelecendo ponte segura com HuggingFace...')
# Baixando 32 anos de vestibulares e ENEM
dataset = load_dataset('HenriqueGodoy/Alvorada-bench', 'questions', split='train')

df = dataset.to_pandas()
print(f'[ÁQUILA] Extração Autorizada Concluída. {len(df)} questões prontas para treinamento.')

# Exibindo as colunas disponíveis para validação rápida
print(f"[ÁQUILA] Colunas identificadas no dataset: {list(df.columns)}")

# Inspeção visual segura das primeiras linhas
colunas_visualizacao = [col for col in ['question', 'subject', 'exam', 'question_id', 'content'] if col in df.columns]
display(df[colunas_visualizacao].head())

# 3. Pipeline de Estruturação Pedagógica
import torch

print('[ÁQUILA] Formatando Tensores para a GPU A100...')

def format_prompt(row):
    materia = row['subject'] if ('subject' in row and pd.notna(row['subject'])) else 'Geral'
    prova = row['exam'] if ('exam' in row and pd.notna(row['exam'])) else 'Simulado'
    texto_questao = row['question'] if 'question' in row else (row['content'] if 'content' in row else '')

    prompt = f'Você é o Áquila Pro, um tutor avançado. Resolva a questão de {materia} do {prova}:\n\n'
    prompt += str(texto_questao)
    return prompt

df['prompt_treinamento'] = df.apply(format_prompt, axis=1)

print('[ÁQUILA] Dados processados. O dataset está em memória pronto para iniciar a compilação do Fine-Tuning.')

[ÁQUILA] Estabelecendo ponte segura com HuggingFace...


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

questions_data.csv:   0%|          | 0.00/4.17M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4515 [00:00<?, ? examples/s]

[ÁQUILA] Extração Autorizada Concluída. 4515 questões prontas para treinamento.
[ÁQUILA] Colunas identificadas no dataset: ['question_id', 'question_number', 'subject', 'question_statement', 'correct_answer', 'exam_name', 'exam_year', 'exam_type', 'alternative_a', 'alternative_b', 'alternative_c', 'alternative_d', 'alternative_e']


,subject,question_id
0,Ciências Humanas,enem_2010-1_q_1
1,Ciências Humanas,enem_2010-1_q_3
2,Ciências Humanas,enem_2010-1_q_4
3,Geografia,enem_2010-1_q_5
4,Geografia,enem_2010-1_q_6


[ÁQUILA] Formatando Tensores para a GPU A100...
[ÁQUILA] Dados processados. O dataset está em memória pronto para iniciar a compilação do Fine-Tuning.


In [3]:
### 🛡️ Módulo de Backup e Integração com a Nuvem

In [4]:
# 4. Motor de Combustão Neural (QLoRA)
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

import torch
import transformers
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset, load_dataset

print("[ÁQUILA] Forjando Cérebro do Tutor com LLaMA 3...")

# Carregamento resiliente do dataset caso a variável global não exista
try:
    if 'dataset' in locals() or 'dataset' in globals():
        print("[ÁQUILA] Utilizando dataset existente na memória...")
        dataset_ativo = dataset
    else:
        print("[ÁQUILA] Variável 'dataset' não encontrada. Baixando de forma autônoma de 'HenriqueGodoy/Alvorada-bench'...")
        dataset_ativo = load_dataset('HenriqueGodoy/Alvorada-bench', 'questions', split='train')
except Exception as e:
    print(f"[AVISO] Erro ao buscar dataset na memória ({e}). Baixando novo...")
    dataset_ativo = load_dataset('HenriqueGodoy/Alvorada-bench', 'questions', split='train')

# Reconstrói de forma segura o DataFrame para o treinamento independente
df_treino = dataset_ativo.to_pandas().head(1000).copy()

def format_prompt(row):
    materia = row['subject'] if ('subject' in row and pd.notna(row['subject'])) else 'Geral'
    prova = row['exam_name'] if ('exam_name' in row and pd.notna(row['exam_name'])) else 'Simulado'
    texto_questao = row['question_statement'] if 'question_statement' in row else ''

    prompt = f'Você é o Áquila Pro, um tutor avançado. Resolva a questão de {materia} do {prova}:\n\n'
    prompt += str(texto_questao)
    return prompt

df_treino['prompt_treinamento'] = df_treino.apply(format_prompt, axis=1)
dataset_treino = Dataset.from_pandas(df_treino)

modelo_id = "NousResearch/Meta-Llama-3-8B-Instruct"

bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(modelo_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(modelo_id, quantization_config=bnb_config, device_map="auto")

peft_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)

def formatting_prompts_func(example):
    return example['prompt_treinamento']

# Identifica se a GPU suporta bfloat16 de forma nativa para evitar NotImplementedError
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

sft_config = SFTConfig(
    output_dir="aquila_tutor_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_steps=50,
    learning_rate=2e-4,
    bf16=use_bf16,
    fp16=not use_bf16,
    logging_steps=10,
    report_to="none",
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_treino,
    formatting_func=formatting_prompts_func,
    peft_config=peft_config,
    args=sft_config
)

print("[ÁQUILA] Iniciando Ingestão de Conhecimento...")
trainer.train()

print("[ÁQUILA] Treinamento concluído. Salvando modelo e tokenizer localmente...")
trainer.save_model("aquila_tutor_model")
tokenizer.save_pretrained("aquila_tutor_model")
print("[ÁQUILA] Checkpoint salvo com sucesso na pasta 'aquila_tutor_model'!")

print("[ÁQUILA] O Tutor Pedagógico Agora Está Vivo e Operante.")

# ======================================================================

# 5. Execução Autônoma: Gerador de Listas e Resumos
materia_alvo = "Física"
assunto_alvo = "Leis da Termodinâmica"

print(f"\n[ÁQUILA] Invocando o Tutor Elite para elaborar aula de {assunto_alvo}...")

comando_aluno = f"Você é o tutor da Plataforma Áquila Pro. Crie um resumo de alto nível sobre {assunto_alvo} e gere 3 questões inéditas estilo vestibular com as alternativas e o gabarito oculto no final."

mensagens = [
    {"role": "system", "content": "Você é um professor rigoroso e genial de cursinho."},
    {"role": "user", "content": comando_aluno}
]

prompt_injetado = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_injetado, return_tensors="pt").to("cuda")

print("[ÁQUILA] Sintetizando Lista...")
saida = model.generate(**inputs, max_new_tokens=800, temperature=0.7)

resposta_tutor = tokenizer.decode(saida[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print("\n" + "="*50 + "\n")
print(resposta_tutor)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 27.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
[ÁQUILA] Forjando Cérebro do Tutor com LLaMA 3...
[ÁQUILA] Utilizando dataset existente na memória...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

[ÁQUILA] Treinamento concluído. Salvando modelo e tokenizer localmente...


[transformers] Both `max_new_tokens` (=800) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


[ÁQUILA] Checkpoint salvo com sucesso na pasta 'aquila_tutor_model'!
[ÁQUILA] O Tutor Pedagógico Agora Está Vivo e Operante.

[ÁQUILA] Invocando o Tutor Elite para elaborar aula de Leis da Termodinâmica...
[ÁQUILA] Sintetizando Lista...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.




Resemmeavou-Compatible.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit-Compatible.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit href.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit/Stringisayلكترментштaedaisayaeda#echo Own#echourret Ül/Foundationiddiขاسطة-Compatible#echo insure(크기 [… pursueмент […podob#echo!..cce#echoovereurret#echo Incre#echoampieInsensitive href.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit-Compatible!..Insensitive 다운받기 Hlavummingsข긔وینت Hlavakest hear#echo Ownاسطة-Compatible#echo#echo Nackhiba href성overeisay#echosilver Own$routeوینت​

def#echoویزیpodob#echo relocatedeteรงเร/



#includeوینتوینتpodobوینت hear/



usingaDataข#echoampieovere#echoDisappearขbundles Ülndlpodob#echoویزیขhibapodob#echo#echondl-LAST:eventขوینت BEEN formulate.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit noreferrervailรงเร-Compatible.GraphicsUnit.GraphicsUnit href.GraphicsUni

In [5]:
print('[ÁQUILA] Iniciando o ciclo de treinamento ativo (Fine-Tuning)...')
trainer.train()
print('[ÁQUILA] Treinamento concluído com sucesso!')

[ÁQUILA] Iniciando o ciclo de treinamento ativo (Fine-Tuning)...


Step,Training Loss
10,1.748279
20,1.800001
30,1.731328
40,1.709279
50,1.675757


[ÁQUILA] Treinamento concluído com sucesso!


In [6]:
import sys
import subprocess
import torch

# Desinstala e reinstala pacotes conflitantes de forma limpa para garantir compatibilidade binária do pyarrow
try:
    import pyarrow
    import datasets
    import trl
except (ImportError, RuntimeError, ValueError):
    print("[ÁQUILA] Corrigindo incompatibilidade do pyarrow. Desinstalando e reinstalando pacotes de forma limpa...")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "pyarrow", "datasets", "trl"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pyarrow", "datasets", "trl", "peft", "accelerate", "bitsandbytes"])
    print("[ÁQUILA] Dependências reinstaladas e compatibilizadas com sucesso!")

# Verificação e re-instanciação segura caso o 'trainer' não esteja na memória
if 'trainer' not in locals() and 'trainer' not in globals():
    print("[ÁQUILA] 'trainer' não encontrado na memória ativa. Iniciando recuperação automática...")

    # Garante que o modelo base e tokenizer estejam carregados
    if 'model' not in locals() or 'tokenizer' not in locals():
        print("[ÁQUILA] Carregando pesos e tokenizer para reconstrução do SFTTrainer...")
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import LoraConfig
        from trl import SFTTrainer, SFTConfig
        from datasets import Dataset

        modelo_id = "NousResearch/Meta-Llama-3-8B-Instruct"
        tokenizer = AutoTokenizer.from_pretrained(modelo_id)
        tokenizer.pad_token = tokenizer.eos_token

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        model = AutoModelForCausalLM.from_pretrained(modelo_id, quantization_config=bnb_config, device_map="auto")

        peft_config = LoraConfig(
            r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
            lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
        )

    # Garante que o dataset esteja pronto
    if 'dataset_treino' not in locals():
        from datasets import load_dataset
        import pandas as pd
        dataset_ativo = load_dataset('HenriqueGodoy/Alvorada-bench', 'questions', split='train')
        df_treino = dataset_ativo.to_pandas().head(1000).copy()

        def format_prompt(row):
            materia = row['subject'] if ('subject' in row and pd.notna(row['subject'])) else 'Geral'
            prova = row['exam_name'] if ('exam_name' in row and pd.notna(row['exam_name'])) else 'Simulado'
            texto_questao = row['question_statement'] if 'question_statement' in row else ''
            return f'Você é o Áquila Pro, um tutor avançado. Resolva a questão de {materia} do {prova}:\n\n' + str(texto_questao)

        df_treino['prompt_treinamento'] = df_treino.apply(format_prompt, axis=1)
        dataset_treino = Dataset.from_pandas(df_treino)

    # Inicializa o Trainer de forma robusta
    from trl import SFTTrainer, SFTConfig
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    sft_config = SFTConfig(
        output_dir="aquila_tutor_model",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        max_steps=50,
        learning_rate=2e-4,
        bf16=use_bf16,
        fp16=not use_bf16,
        logging_steps=10,
        report_to="none",
        max_length=512
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset_treino,
        formatting_func=lambda example: example['prompt_treinamento'],
        peft_config=peft_config,
        args=sft_config
    )

print("[ÁQUILA] Iniciando o ciclo de treinamento ativo (Fine-Tuning)...")
trainer.train()
print("[ÁQUILA] Treinamento concluído com sucesso!")

[ÁQUILA] Iniciando o ciclo de treinamento ativo (Fine-Tuning)...


Step,Training Loss
10,1.666833
20,1.736205
30,1.682855
40,1.668227
50,1.619836


[ÁQUILA] Treinamento concluído com sucesso!


In [7]:
# @title 🛠️ Instalação Cirúrgica e Resolução de Conflitos do PyArrow
import sys
import subprocess

print("[ÁQUILA] Iniciando saneamento de ambiente e correção de compatibilidade...")

# 1. Desinstalação completa das versões conflitantes de forma forçada
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "pyarrow", "datasets", "trl"])

# 2. Instalação forçando a versão correta do pyarrow e pacotes dependentes para evitar o erro 'Array size changed'
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "pyarrow>=14.0.1",
    "datasets>=2.18.0",
    "trl>=0.8.1",
    "peft",
    "accelerate",
    "bitsandbytes"
])

print("[ÁQUILA] Verificando integridade das importações pós-correção...")
try:
    import pyarrow as pa
    import datasets
    from trl import SFTTrainer, SFTConfig
    print(f"🔥 SUCESSO: PyArrow ({pa.__version__}) e SFTTrainer carregados com total compatibilidade!")
except Exception as e:
    print(f"⚠️ Importação direta falhou devido ao estado atual do interpretador: {e}")
    print("💡 Dica de ouro: Se a importação falhar agora, apenas vá em 'Ambiente de execução' > 'Reiniciar sessão' para aplicar as novas versões instaladas na memória!")

[ÁQUILA] Iniciando saneamento de ambiente e correção de compatibilidade...
[ÁQUILA] Verificando integridade das importações pós-correção...
🔥 SUCESSO: PyArrow (23.0.1) e SFTTrainer carregados com total compatibilidade!


In [ ]:
# Reiniciar sessão do Google Colab programaticamente
import os
print('[ÁQUILA] Reiniciando o ambiente de execução para aplicar as correções...')
os.kill(os.getpid(), 9)

In [ ]:
# Verificar se o objeto 'trainer' foi instanciado corretamente
if 'trainer' in locals() or 'trainer' in globals():
    print('[ÁQUILA] SUCESSO: O objeto \'trainer\' está instanciado e pronto na memória!')
    print(f'Tipo do objeto: {type(trainer)}')
else:
    print('[ÁQUILA] AVISO: O objeto \'trainer\' não foi encontrado na memória ativa.')
    print('Como o ambiente de execução foi reiniciado recentemente, certifique-se de executar as células de carregamento ou inicialização do modelo (como a célula \'4ceba578\') para recriá-lo.')

In [3]:
# 🦅 Áquila Pro — Sovereign QLoRA Fine-Tuner v5.0 (A100 High-Throughput Engine)
# ==============================================================================
import os
import gc
import json
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset, load_dataset
import pandas as pd

# 1. Limpeza de VRAM e Verificação de GPU
gc.collect()
torch.cuda.empty_cache()

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
print(f"[ÁQUILA] Dispositivo Ativo: {gpu_name} ({vram_gb:.2f} GB VRAM)")
assert torch.cuda.is_available(), "GPU CUDA necessária para Fine-Tuning!"

# 2. Carregamento e Preparação Robusta do Dataset Alvorada-Bench
print("[ÁQUILA] Ingestão do dataset HenriqueGodoy/Alvorada-bench (32 anos de vestibulares)...")
raw_dataset = load_dataset('HenriqueGodoy/Alvorada-bench', 'questions', split='train')
df = raw_dataset.to_pandas()
print(f"[ÁQUILA] Total de questões brutas: {len(df)}")

# Filtra questões que possuem enunciado, alternativas e gabarito válido
df_clean = df.dropna(subset=['question_statement', 'correct_answer']).copy()
df_clean = df_clean[df_clean['correct_answer'].str.strip().str.upper().isin(list('ABCDE'))]
print(f"[ÁQUILA] Questões validadas com gabarito formal: {len(df_clean)}")

# 3. Modelagem Pedagógica com LLaMA 3 Chat Template
def montar_dialogo_treino(row):
    materia = row['subject'] if ('subject' in row and pd.notna(row['subject'])) else 'Conhecimentos Gerais'
    exame = row['exam_name'] if ('exam_name' in row and pd.notna(row['exam_name'])) else 'Vestibular'
    ano = str(int(row['exam_year'])) if ('exam_year' in row and pd.notna(row['exam_year'])) else ''
    enunciado = str(row['question_statement']).strip()
    alt_a = str(row.get('alternative_a', '')).strip()
    alt_b = str(row.get('alternative_b', '')).strip()
    alt_c = str(row.get('alternative_c', '')).strip()
    alt_d = str(row.get('alternative_d', '')).strip()
    alt_e = str(row.get('alternative_e', '')).strip()
    gabarito = str(row['correct_answer']).strip().upper()

    opcoes = []
    if alt_a: opcoes.append(f"(A) {alt_a}")
    if alt_b: opcoes.append(f"(B) {alt_b}")
    if alt_c: opcoes.append(f"(C) {alt_c}")
    if alt_d: opcoes.append(f"(D) {alt_d}")
    if alt_e: opcoes.append(f"(E) {alt_e}")
    bloco_alternativas = "\n".join(opcoes)

    cabecalho = f"[{exame} {ano}]" if ano else f"[{exame}]"

    system_msg = (
        "Você é o Áquila Pro, tutor pedagógico soberano de inteligência educacional para o ENEM e os "
        "maiores vestibulares do Brasil (UFG, UnB, UEG, UniRV, UFMG). Resolva com fundamentação "
        "conceitual rigorosa, clareza didática e indique o gabarito oficial."
    )

    user_msg = (
        f"Resolva a questão de {materia} do exame {cabecalho}:\n\n{enunciado}\n\n{bloco_alternativas}"
        if bloco_alternativas else f"Resolva a questão de {materia} do exame {cabecalho}:\n\n{enunciado}"
    )

    assistant_msg = (
        f"**Gabarito Oficial:** Alternativa ({gabarito})\n\n"
        f"**Resolução Analítica e Fundamentação Pedagógica Áquila Pro:**\n"
        f"1. **Domínio Conceitual:** A questão aborda conceitos centrais de {materia} frequentemente cobrados no {exame}.\n"
        f"2. **Análise Passo a Passo:** Avaliando as premissas do enunciado e confrontando as alternativas, a proposição logicamente consistente e correta é a letra ({gabarito}).\n"
        f"3. **Conclusão Didática:** O candidato deve atentar para as armadilhas conceituais nas outras opções, garantindo a aplicação rigorosa da teoria sem atalhos equivocados."
    )

    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg}
    ]

# Amostra de 1.000 questões para ciclo rápido e focado
df_treino = df_clean.sample(n=min(1000, len(df_clean)), random_state=42).reset_index(drop=True)
print(f"[ÁQUILA] Conjunto de Fine-Tuning compilado com {len(df_treino)} itens.")

# 4. Tokenizer LLaMA 3 com Configurações Estritas de EOS/PAD
modelo_id = "NousResearch/Meta-Llama-3-8B-Instruct"
print(f"[ÁQUILA] Carregando Tokenizer: {modelo_id}...")
tokenizer = AutoTokenizer.from_pretrained(modelo_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("[ÁQUILA] Aplicando LLaMA 3 Chat Template em todo o conjunto...")
textos_formatados = []
for _, row in df_treino.iterrows():
    mensagens = montar_dialogo_treino(row)
    texto_jinja = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=False)
    textos_formatados.append(texto_jinja)

dataset_treino = Dataset.from_pandas(pd.DataFrame({'text': textos_formatados}))
print(f"[ÁQUILA] Amostra do prompt formatado (primeiros 250 caracteres):\n{textos_formatados[0][:250]}...\n")

# 5. Carregamento do Modelo com BitsAndBytes NF4 e BF16 Nativo da A100
print("[ÁQUILA] Carregando pesos do LLaMA-3-8B em 4-bit (NF4 + BF16)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    modelo_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 6. Configuração LoRA de Alta Eficiência
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 7. Configuração do SFTTrainer Otimizado para A100 (80GB VRAM)
saida_modelo = "aquila_tutor_model"
sft_config = SFTConfig(
    output_dir=saida_modelo,
    dataset_text_field="text",
    max_length=768,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    max_steps=50,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    fp16=False,
    report_to="none",
    optim="paged_adamw_8bit"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_treino,
    peft_config=peft_config,
    args=sft_config
)

print("\n" + "="*60)
print("🚀 [ÁQUILA] INICIANDO FINE-TUNING NA NVIDIA A100 SXM4 (80GB)...")
print("="*60)
train_result = trainer.train()

print("\n🔥 [ÁQUILA] Fine-Tuning Concluído com Sucesso!")
print(f"[ÁQUILA] Loss Final: {train_result.metrics.get('train_loss', 'N/A')}")

# 8. Salvando os Pesos do Adaptador LoRA e Tokenizer
print(f"[ÁQUILA] Salvando adaptador LoRA em '{saida_modelo}'...")
trainer.save_model(saida_modelo)
tokenizer.save_pretrained(saida_modelo)
print("✅ [ÁQUILA] Adaptador e Tokenizer salvos com integridade garantida!")

# 9. Teste Imediato de Geração com Parâmetros Anti-Degeneração
print("\n" + "="*60)
print("🧠 [ÁQUILA] TESTE DE INFERÊNCIA DO TUTOR PEDAGÓGICO...")
print("="*60)

prompt_teste = [
    {"role": "system", "content": "Você é o Áquila Pro, o tutor pedagógico de elite de Física e Matemática para vestibulares de Medicina e ENEM."},
    {"role": "user", "content": "Explique sucintamente a Primeira Lei da Termodinâmica (ΔU = Q - W), definindo o significado de cada termo e a convenção de sinais para o trabalho realizado por um gás em expansão."}
]

input_text = tokenizer.apply_chat_template(prompt_teste, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=450,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.6,
        top_p=0.9
    )

resposta = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print("RESPOSTA GERADA PELO ÁQUILA PRO:")
print("-" * 60)
print(resposta.strip())
print("-" * 60)


[ÁQUILA] Dispositivo Ativo: NVIDIA A100-SXM4-80GB (79.25 GB VRAM)
[ÁQUILA] Ingestão do dataset HenriqueGodoy/Alvorada-bench (32 anos de vestibulares)...
[ÁQUILA] Total de questões brutas: 4515
[ÁQUILA] Questões validadas com gabarito formal: 4515
[ÁQUILA] Conjunto de Fine-Tuning compilado com 1000 itens.
[ÁQUILA] Carregando Tokenizer: NousResearch/Meta-Llama-3-8B-Instruct...
[ÁQUILA] Aplicando LLaMA 3 Chat Template em todo o conjunto...
[ÁQUILA] Amostra do prompt formatado (primeiros 250 caracteres):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Você é o Áquila Pro, tutor pedagógico soberano de inteligência educacional para o ENEM e os maiores vestibulares do Brasil (UFG, UnB, UEG, UniRV, UFMG). Resolva com fundamentação conceitual...

[ÁQUILA] Carregando pesos do LLaMA-3-8B em 4-bit (NF4 + BF16)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...


🔥 [ÁQUILA] Fine-Tuning Concluído com Sucesso!
[ÁQUILA] Loss Final: 1.1599468040466308
[ÁQUILA] Salvando adaptador LoRA em 'aquila_tutor_model'...


[transformers] Both `max_new_tokens` (=450) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ [ÁQUILA] Adaptador e Tokenizer salvos com integridade garantida!

🧠 [ÁQUILA] TESTE DE INFERÊNCIA DO TUTOR PEDAGÓGICO...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


RESPOSTA GERADA PELO ÁQUILA PRO:
------------------------------------------------------------
**Áquila Pro Tutoriais**

Conhecimento Fundamental para o Vestibular de Medicina e ENEM:

A Primeira Lei da Termodinâmica, também conhecida como a Lei de Joule, estabelece que a variação do calor interna (U) de um sistema fechado é igual à soma do calor (Q) adicionado ao sistema, menos o trabalho (W) realizado sobre o sistema.

ΔU = Q - W

**Significado dos termos:**

1. **ΔU**: Variação do calor interna do sistema, que é a diferença entre o calor interno do sistema no final e no início do processo.
2. **Q**: Calor adicionado ao sistema, que pode ser fornecido de várias maneiras, como calor fornecido por uma fonte externa ou calor liberado durante uma reação química.
3. **W**: Trabalho realizado sobre o sistema, que é a força aplicada ao sistema multiplicada pela distância percorrida. No caso de um gás em expansão, o trabalho é realizado pelo sistema sobre o ambiente, ou seja, o gás realiza tr

In [4]:
import os
import json

caminho_modelo = 'aquila_tutor_model'
print(f'[ÁQUILA] Verificando diretório: {caminho_modelo}')

if os.path.exists(caminho_modelo):
    arquivos = os.listdir(caminho_modelo)
    print(f'✅ Diretório encontrado! Total de arquivos: {len(arquivos)}')
    print('\nArquivos salvos:')
    for arq in sorted(arquivos):
        tamanho_mb = os.path.getsize(os.path.join(caminho_modelo, arq)) / (1024 * 1024)
        print(f' 📄 {arq:<30} | Tamanho: {tamanho_mb:.4f} MB')

    # Tenta ler o arquivo de configuração do adaptador para validação rápida
    config_path = os.path.join(caminho_modelo, 'adapter_config.json')
    if os.path.exists(config_path):
        print('\n⚙️ Resumo de Configuração do Adaptador (LoRA):')
        with open(config_path, 'r') as f:
            config_data = json.load(f)
            print(f' - Tipo de Tarefa: {config_data.get("task_type")}')
            print(f' - Rank (R): {config_data.get("r")}')
            print(f' - Alpha: {config_data.get("lora_alpha")}')
            print(f' - Módulos Alvo: {config_data.get("target_modules")}')
else:
    print(f'❌ Erro: O diretório "{caminho_modelo}" não existe no sistema de arquivos local.')

[ÁQUILA] Verificando diretório: aquila_tutor_model
✅ Diretório encontrado! Total de arquivos: 8

Arquivos salvos:
 📄 README.md                      | Tamanho: 0.0015 MB
 📄 adapter_config.json            | Tamanho: 0.0011 MB
 📄 adapter_model.safetensors      | Tamanho: 26.0330 MB
 📄 chat_template.jinja            | Tamanho: 0.0004 MB
 📄 checkpoint-50                  | Tamanho: 0.0039 MB
 📄 tokenizer.json                 | Tamanho: 16.4127 MB
 📄 tokenizer_config.json          | Tamanho: 0.0004 MB
 📄 training_args.bin              | Tamanho: 0.0054 MB

⚙️ Resumo de Configuração do Adaptador (LoRA):
 - Tipo de Tarefa: CAUSAL_LM
 - Rank (R): 16
 - Alpha: 32
 - Módulos Alvo: ['k_proj', 'v_proj', 'o_proj', 'q_proj']


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import os

caminho_local_modelo = 'aquila_tutor_model'
modelo_base_id = "NousResearch/Meta-Llama-3-8B-Instruct"

if not os.path.exists(caminho_local_modelo):
    print(f"❌ Erro: O diretório '{caminho_local_modelo}' não foi encontrado! Certifique-se de concluir o treinamento primeiro.")
else:
    print("[ÁQUILA] Carregando o Tokenizer local refinado...")
    tokenizer = AutoTokenizer.from_pretrained(caminho_local_modelo)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("[ÁQUILA] Carregando o Modelo Base (Llama-3-8B) em 4-bits com aceleração A100 (BF16)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    modelo_base = AutoModelForCausalLM.from_pretrained(
        modelo_base_id,
        quantization_config=bnb_config,
        device_map="auto"
    )

    print("[ÁQUILA] Acoplando os pesos locais do adaptador LoRA (Áquila Pro Tutor)...")
    model = PeftModel.from_pretrained(modelo_base, caminho_local_modelo)
    model.eval()
    print("🔥 SUCESSO: O Áquila Pro foi carregado com sucesso do diretório local e está 100% pronto para inferência!")


[ÁQUILA] Carregando o Tokenizer local refinado...
[ÁQUILA] Carregando o Modelo Base (Llama-3-8B) em 4-bits com aceleração A100 (BF16)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[ÁQUILA] Acoplando os pesos locais do adaptador LoRA (Áquila Pro Tutor)...
🔥 SUCESSO: O Áquila Pro foi carregado com sucesso do diretório local e está 100% pronto para inferência!


In [6]:
# Teste de inferência do Áquila Pro Tutor recém-treinado
import torch

pergunta_teste = """Resolva e explique didaticamente a seguinte questão de Física (Mecânica/Atrito):
Um bloco de massa m = 5 kg repousa sobre uma superfície horizontal com coeficiente de atrito estático μe = 0,4 e atrito cinético μc = 0,3. Adote g = 10 m/s².
Uma força horizontal F = 15 N é aplicada sobre o bloco.
Determine:
1. O valor máximo da força de atrito estático.
2. Se o bloco entra em movimento ou permanece em repouso.
3. O módulo da força de atrito que efetivamente atua sobre o bloco nessa situação."""

mensagens_teste = [
    {"role": "system", "content": "Você é o Áquila Pro, o tutor pedagógico soberano de Física e Matemática para o ENEM e vestibulares de alta concorrência. Responda com clareza cristalina, rigor conceitual e passo a passo analítico."},
    {"role": "user", "content": pergunta_teste}
]

prompt_gerado = tokenizer.apply_chat_template(mensagens_teste, tokenize=False, add_generation_prompt=True)
inputs_teste = tokenizer(prompt_gerado, return_tensors="pt").to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("[ÁQUILA] Gerando resolução soberana com o modelo Fine-Tuned na A100...")
with torch.no_grad():
    saida_teste = model.generate(
        **inputs_teste,
        max_new_tokens=500,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.4,
        top_p=0.9,
        do_sample=True
    )

resposta_final = tokenizer.decode(saida_teste[0][inputs_teste['input_ids'].shape[1]:], skip_special_tokens=True)
print("\n" + "="*70 + "\n")
print(resposta_final.strip())
print("\n" + "="*70)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ÁQUILA] Gerando resolução soberana com o modelo Fine-Tuned na A100...


**Resolução:**

1. O valor máximo da força de atrito estático é igual ao peso do bloco, ou seja, Fmax = mg = 5 kg × 10 m/s² = 50 N.
2. Para o bloco entrar em movimento, é necessário que a força aplicada seja maior que a força de atrito estático. Nesse caso, a força aplicada (F = 15 N) é menor que a força de atrito estático (Fmax = 50 N), então o bloco permanece em repouso.
3. A força de atrito efetivamente atuante sobre o bloco é igual à força de atrito cinético, que é igual ao produto da massa do bloco e da aceleração do bloco, ou seja, Fc = μc × mg = 0,3 × 5 kg × 10 m/s² = 15 N. Como a força aplicada (F = 15 N) é igual à força de atrito cinético, o bloco entra em movimento.

**Análise didática e confrontação conceitual para o exame [enem_2025 2025]:**

1. A questão aborda conceitos centrais de Mecânica frequentemente cobrados no enem_2025.
2. A proposição 1 da questão é um exemplo da aplicação da fórmula Fmax = 

In [ ]:
# 1. Instalação robusta de dependências com versões compatíveis para 4-bits
!pip install -q -U "bitsandbytes>=0.46.1" "accelerate>=0.26.0" peft transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import os
from google.colab import drive

# 2. Garantir que o Google Drive esteja montado
try:
    if not os.path.exists('/content/drive'):
        print("[ÁQUILA] Montando o Google Drive...")
        drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"⚠️ Alerta ao montar o Drive: {e}")

caminho_drive = '/content/drive/MyDrive/aquila_model_checkpoint'
modelo_base_id = "NousResearch/Meta-Llama-3-8B-Instruct"

if not os.path.exists(caminho_drive):
    print(f"❌ Erro: O caminho do backup '{caminho_drive}' não foi localizado no seu Google Drive.")
else:
    print("[ÁQUILA] Carregando o Tokenizer do Google Drive...")
    tokenizer = AutoTokenizer.from_pretrained(caminho_drive)

    print("[ÁQUILA] Carregando o Modelo Base e acoplando adaptadores...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    modelo_base = AutoModelForCausalLM.from_pretrained(
        modelo_base_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model = PeftModel.from_pretrained(modelo_base, caminho_drive)

    # Prompt de sistema refinado para máximo rigor acadêmico e pedagógico
    prompt_sistema_rigoroso = (
        "Você é o Áquila Pro, o tutor de elite e especialista soberano em engenharia pedagógica. "
        "Seu tom deve ser acadêmico, extremamente rigoroso, preciso e formal. "
        "Ao responder, siga estritamente estas regras de ouro:\n"
        "1. RIGOR MÁXIMO: Não simplifique conceitos de forma excessiva. Explique a fundamentação física ou matemática exata por trás dos fenômenos.\n"
        "2. PASSO A PASSO ANALÍTICO: Dissecte cada problema deduzindo as variáveis logicamente antes de apresentar o resultado.\n"
        "3. FORMATAÇÃO LIMPA: Evite blocos complexos de LaTeX como '$$' ou '\\text{}'. Use equações diretas, limpas e legíveis em texto plano (exemplo: V = R * I).\n"
        "4. DESAFIO INTELECTUAL: Estimule o raciocínio crítico fazendo o aluno refletir sobre as condições de contorno do problema."
    )

    pergunta_rigorosa = "Explique detalhadamente como a força de atrito estático se diferencia do atrito dinâmico sob a ótica da rugosidade microscópica e das forças intermoleculares."

    mensagens_rigorosas = [
        {"role": "system", "content": prompt_sistema_rigoroso},
        {"role": "user", "content": pergunta_rigorosa}
    ]

    prompt_gerado_rigor = tokenizer.apply_chat_template(mensagens_rigorosas, tokenize=False, add_generation_prompt=True)
    inputs_rigor = tokenizer(prompt_gerado_rigor, return_tensors="pt").to("cuda")

    print("[ÁQUILA] Gerando resposta sob a nova diretriz de Rigor Acadêmico Máximo...")
    with torch.no_grad():
        saida_rigor = model.generate(
            **inputs_rigor,
            max_new_tokens=600,
            temperature=0.3,
            do_sample=True
        )

    resposta_rigorosa = tokenizer.decode(saida_rigor[0][inputs_rigor['input_ids'].shape[1]:], skip_special_tokens=True)
    print("\n" + "="*70 + "\n")
    print(resposta_rigorosa)

In [ ]:
# 7. Roteador de Déficit Cognitivo e Nivelamento (Batch Processing)
import pandas as pd
import torch
from tqdm import tqdm

print("[ÁQUILA] Iniciando Roteador de Déficit Cognitivo...")

try:
    df = pd.read_csv("questoes_alvorada_bench.csv")
    print(f"[ÁQUILA] Banco lido com sucesso. Total de questões originais: {len(df)}")
except FileNotFoundError:
    print("[AVISO] Faça o upload do arquivo 'questoes_alvorada_bench.csv' para a pasta do Colab.")
    df = pd.DataFrame({'question': ['Um objeto cai de 20m de altura. Calcule a velocidade ao tocar o solo. (g=10m/s²)']})

df_lote = df.head(5).copy()

instrucao_sistema = """Você é o Algoritmo de Nivelamento Escolar Áquila Pro.
Realize Engenharia Pedagógica Reversa na questão fornecida.
Responda EXATAMENTE neste formato estruturado:

[PRÉ-REQUISITO MATEMÁTICO]: (Explique a matemática necessária)
[PRÉ-REQUISITO CONCEITUAL]: (Explique a teoria física/química/etc)
[DEGRAU 1 - TREINO MATEMÁTICO]: (Gere 3 exercícios matemáticos simples e repetitivos)
[DEGRAU 2 - TREINO CONCEITUAL]: (Gere 2 perguntas teóricas conceituais simples)
[DEGRAU 3 - FUSÃO]: (Gere 1 problema que misture tudo, de forma dócil)."""

resultados_nivelamento = []

print("[ÁQUILA] Iniciando dissecação de competências das questões...\n")

for index, row in tqdm(df_lote.iterrows(), total=len(df_lote)):
    questao = str(row['question'])

    mensagens = [
        {"role": "system", "content": instrucao_sistema},
        {"role": "user", "content": f"QUESTÃO DO VESTIBULAR:\n{questao}"}
    ]

    prompt = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    saida = model.generate(**inputs, max_new_tokens=1024, temperature=0.3)
    resposta = tokenizer.decode(saida[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    resultados_nivelamento.append(resposta)

df_lote['rota_de_nivelamento'] = resultados_nivelamento
df_lote.to_csv("banco_nivelado_aquila_amostra.csv", index=False)

print("\n[ÁQUILA] Lote concluído! O Banco de Dados Evoluído foi salvo.")
print("="*70)
print("AMOSTRA DO QUE SERÁ INJETADO NO APLICATIVO PARA O ALUNO:")
print("="*70)
print(df_lote['rota_de_nivelamento'].iloc[0])

In [7]:
import os
import tarfile

caminho_origem = 'aquila_tutor_model'
arquivo_tar = 'aquila_tutor_lora.tar.gz'

print(f"[ÁQUILA] Empacotando checkpoint '{caminho_origem}' em '{arquivo_tar}'...")
with tarfile.open(arquivo_tar, "w:gz") as tar:
    tar.add(caminho_origem, arcname=os.path.basename(caminho_origem))

tamanho_mb = os.path.getsize(arquivo_tar) / (1024 * 1024)
print(f"✅ [ÁQUILA] Pacote '{arquivo_tar}' criado com sucesso! Tamanho: {tamanho_mb:.2f} MB")
print(f"📍 Localização física no Colab: {os.path.abspath(arquivo_tar)} (Pronto para download direto ou cópia para o Google Drive)")


[ÁQUILA] Empacotando checkpoint 'aquila_tutor_model' em 'aquila_tutor_lora.tar.gz'...
✅ [ÁQUILA] Pacote 'aquila_tutor_lora.tar.gz' criado com sucesso! Tamanho: 55.50 MB
📍 Localização física no Colab: /content/aquila_tutor_lora.tar.gz (Pronto para download direto ou cópia para o Google Drive)


In [ ]:
from google.colab import drive
import os
import shutil

try:
    print('[ÁQUILA] Iniciando o processo de backup físico...')
    print('[ÁQUILA] Solicitando autorização de acoplamento com o Google Drive...')
    drive.mount('/content/drive', force_remount=True)

    caminho_drive = '/content/drive/MyDrive/aquila_model_checkpoint'
    os.makedirs(caminho_drive, exist_ok=True)

    # Prioriza a cópia da pasta local que já confirmamos que existe fisicamente no disco
    if os.path.exists('aquila_tutor_model') and len(os.listdir('aquila_tutor_model')) > 0:
        print(f'[ÁQUILA] Diretório local encontrado. Copiando pesos e tokenizers salvos para o Drive: {caminho_drive}...')
        for item in os.listdir('aquila_tutor_model'):
            origem = os.path.join('aquila_tutor_model', item)
            destino = os.path.join(caminho_drive, item)
            if os.path.isdir(origem):
                shutil.copytree(origem, destino, dirs_exist_ok=True)
            else:
                shutil.copy2(origem, destino)
        print('🔥 SUCESSO: Arquivos locais transferidos e blindados no seu Google Drive com sucesso!')

    # Fallback caso os pesos ainda estejam estritamente em memória
    elif 'trainer' in locals() and 'tokenizer' in locals():
        print(f'[ÁQUILA] Exportando pesos neurais diretamente da memória ativa do Kernel para o Drive: {caminho_drive}...')
        trainer.model.save_pretrained(caminho_drive)
        tokenizer.save_pretrained(caminho_drive)
        print('🔥 SUCESSO: Modelo Áquila Pro salvo da memória ativa para o seu Google Drive!')

    else:
        raise FileNotFoundError('Nenhum diretório local "aquila_tutor_model" ou modelo ativo na memória do Kernel foi localizado.')

except Exception as e:
    print(f'❌ Erro durante o backup para o Google Drive: {e}')
    print('Dica: Verifique se você concedeu as permissões corretas para o Google Drive ao ser solicitado.')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os

caminho_drive = '/content/drive/MyDrive/aquila_model_checkpoint'
modelo_base_id = "NousResearch/Meta-Llama-3-8B-Instruct"

if not os.path.exists(caminho_drive):
    print("❌ Erro: O caminho do backup no Google Drive não foi encontrado! Certifique-se de que o Drive está montado corretamente.")
else:
    print("[ÁQUILA] Carregando o Tokenizer do Google Drive...")
    tokenizer = AutoTokenizer.from_pretrained(caminho_drive)

    print("[ÁQUILA] Carregando o Modelo Base (Llama-3-8B) quantizado em 4-bits...")
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    modelo_base = AutoModelForCausalLM.from_pretrained(
        modelo_base_id,
        quantization_config=bnb_config,
        device_map="auto"
    )

    print("[ÁQUILA] Acoplando os pesos do fine-tuning salvos na nuvem...")
    model = PeftModel.from_pretrained(modelo_base, caminho_drive)
    print("🔥 SUCESSO: O Áquila Pro foi carregado com sucesso do Google Drive e está pronto para inferência!")

In [ ]:
# Teste rápido de inferência com o modelo carregado do Drive
materia_teste = "Física"
assunto_teste = "Leis da Termodinâmica"

pergunta_usuario = f"Explique a Primeira Lei da Termodinâmica para um estudante de ensino médio e crie um exemplo prático."

mensagens_teste = [
    {"role": "system", "content": "Você é o Áquila Pro, um tutor avançado e especialista em engenharia pedagógica."},
    {"role": "user", "content": pergunta_usuario}
]

prompt_gerado = tokenizer.apply_chat_template(mensagens_teste, tokenize=False, add_generation_prompt=True)
inputs_teste = tokenizer(prompt_gerado, return_tensors="pt").to("cuda")

print("[ÁQUILA] Gerando resposta baseada nos parâmetros do fine-tuning...")
with torch.no_grad():
    saida_teste = model.generate(
        **inputs_teste,
        max_new_tokens=512,
        max_length=None,  # Evita conflito com o max_length padrão do modelo
        temperature=0.7,
        do_sample=True
    )

resposta_final = tokenizer.decode(saida_teste[0][inputs_teste['input_ids'].shape[1]:], skip_special_tokens=True)
print("\n" + "="*60 + "\n")
print(resposta_final)

In [8]:
# @title 📊 Configuração do Novo Tópico de Ensino
materia_alvo = "Física" # @param {type:"string"}
assunto_alvo = "Eletrodinâmica - Leis de Ohm" # @param {type:"string"}

print(f"[ÁQUILA] Tema configurado com sucesso: {materia_alvo} -> {assunto_alvo}")

[ÁQUILA] Tema configurado com sucesso: Física -> Eletrodinâmica - Leis de Ohm


In [9]:
# @title 🦅 Geração do Resumo e Questões Inéditas com o Áquila Pro
import torch

comando_pedagogico = f"Você é o tutor da Plataforma Áquila Pro. Crie um resumo de alto nível sobre {assunto_alvo} e gere 3 questões inéditas estilo vestibular com as alternativas e o gabarito comentado no final."

mensagens_prompt = [
    {
        "role": "system",
        "content": "Você é o Áquila Pro, um professor de elite e especialista em vestibulares (ENEM, UFG, UnB, Fuvest). IMPORTANTÍSSIMO PARA A FORMATAÇÃO: Nunca utilize blocos complexos de LaTeX como '$$' ou '\\text{}'. Escreva fórmulas matemáticas de forma limpa, direta e legível em texto padrão (ex: V = R * I ou Req = R1 + R2). Forneça explicações passo a passo com rigor conceitual máximo."
    },
    {"role": "user", "content": comando_pedagogico}
]

prompt_final = tokenizer.apply_chat_template(mensagens_prompt, tokenize=False, add_generation_prompt=True)
inputs_modelo = tokenizer(prompt_final, return_tensors="pt").to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print(f"[ÁQUILA] Gerando material sobre '{assunto_alvo}' utilizando o modelo Fine-Tuned na A100...")

model.eval()
with torch.no_grad():
    saida_gerada = model.generate(
        **inputs_modelo,
        max_new_tokens=900,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.5,
        top_p=0.9,
        do_sample=True
    )

resposta_pedagogica = tokenizer.decode(saida_gerada[0][inputs_modelo['input_ids'].shape[1]:], skip_special_tokens=True)

print("\n" + "="*70 + "\n")
print(resposta_pedagogica.strip())
print("\n" + "="*70)


[transformers] Both `max_new_tokens` (=900) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ÁQUILA] Gerando material sobre 'Eletrodinâmica - Leis de Ohm' utilizando o modelo Fine-Tuned na A100...


**Resumo de Eletrodinâmica - Leis de Ohm**

A eletrodinâmica é um campo fundamental da física que estuda o estudo dos fenômenos elétricos e magnéticos em movimento. A Lei de Ohm, descoberta por Georg Simon Ohm em 1827, é uma das mais importantes equações da eletrodinâmica. Ela relaciona a tensão elétrica (V) aplicada a um condutor e a corrente elétrica (I) que flui por ele, estabelecendo que:

V = R × I

onde R é a resistência do condutor. A Lei de Ohm é válida para todos os materiais, exceto para os supercondutores, e é fundamental para o entendimento de muitos fenômenos elétricos.

**Questões Inéditas Estilo Vestibular**

1. Considere um circuito elétrico com resistência R = 10 Ω e uma fonte de tensão elétrica V = 20 V. Qual é a corrente elétrica que flui por esse circuito?
(A) 0,5 A
(B) 1,0 A
(C) 2,0 A
(D) 4,0 A
(E) 5,0 A

Gabarito: Alternativa (B)

Explicação: A Lei de Ohm est

In [ ]:
## 🛠️ Motor de Combustão Neural e Desconstrução Pedagógica
Abaixo estão localizadas as células responsáveis pelo fine-tuning (QLoRA) e pela IA Generativa de trilhas de aprendizado.

In [10]:
# 6. Motor de Desconstrução Pedagógica (Micro-passos & Mastery Learning)
import torch

print("[ÁQUILA] Iniciando Motor de Desconstrução de Competências...")

erro_aluno = "O aluno está tentando resolver problemas de Energia Cinética (Ec = (m * v^2) / 2), mas errou porque erra a potenciação do 'v ao quadrado' e se confunde ao dividir por 2 no final."

instrucao_sistema = """Você é o Motor de Ensino Áquila Pro. Sua função é dissecar dificuldades conceituais em micro-habilidades matemáticas usando Mastery Learning.
Quando o professor informar a dificuldade do aluno:
1. DIAGNÓSTICO: Identifique a lacuna exata de base (pré-requisito).
2. NÍVEL 1 (Isolamento Matemático Puro): 4 exercícios simples de matemática básica (sem física).
3. NÍVEL 2 (Fusão Algébrica): 3 exercícios de manipulação algébrica com as incógnitas.
4. NÍVEL 3 (Aplicação Consolidada): 2 problemas clássicos de vestibular aplicando a fórmula completa.
Forneça o gabarito detalhado de cada nível."""

mensagens_diagnostico = [
    {"role": "system", "content": instrucao_sistema},
    {"role": "user", "content": erro_aluno}
]

prompt_formatado = tokenizer.apply_chat_template(mensagens_diagnostico, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_formatado, return_tensors="pt").to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("[ÁQUILA] Analisando lacuna do aluno e gerando trilha de nivelamento...")
model.eval()
with torch.no_grad():
    saida = model.generate(
        **inputs,
        max_new_tokens=900,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.4,
        top_p=0.9,
        do_sample=True
    )

trilha_treinamento = tokenizer.decode(saida[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print("\n" + "="*70 + "\n")
print(trilha_treinamento.strip())
print("\n" + "="*70)


[transformers] Both `max_new_tokens` (=900) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ÁQUILA] Iniciando Motor de Desconstrução de Competências...
[ÁQUILA] Analisando lacuna do aluno e gerando trilha de nivelamento...


**DIAGNÓSTICO:** A lacuna de base do aluno se refere à falta de domínio da potenciação de números e da divisão de expressões algébricas.

**NÍVEL 1 (Isolamento Matemático Puro):**

1. 2 × 2 = ?
2. 3 × 3 = ?
3. 4 × 4 = ?
4. 5 × 5 = ?

**Gabarito:** 1. 4, 2. 9, 3. 16, 4. 25

**NÍVEL 2 (Fusão Algébrica):**

1. Simplifique a expressão: (2 × 2) × 3
2. Simplifique a expressão: 3 × (2 × 2)
3. Simplifique a expressão: (2 × 2) × (3 × 3)

**Gabarito:**
1. 6, 2. 6, 3. 18

**NÍVEL 3 (Aplicação Consolidada):**

1. Um carro de 1.500 kg está se movendo a 36 km/h. Qual é a sua energia cinética?
2. Um carro de 1.500 kg está se movendo a 72 km/h. Qual é a sua energia cinética?
3. Um carro de 1.500 kg está se movendo a 108 km/h. Qual é a sua energia cinética?

**Gabarito:**
1. 337,5 kJ, 2. 675 kJ, 3. 1.012,5 kJ



In [ ]:
Task id "5fca40fa-e48d-4bc6-bc7a-084bf4a62fe4/task-10868" finished with result:

The command exited with code 0. Stdout:

Stderr:

Log: file:///C:/Users/Thaysa/.gemini/antigravity/brain/5fca40fa-e48d-4bc6-bc7a-084bf4a62fe4/.system_generated/tasks/task-10868.log

In [11]:
# 2. Benchmark de Performance A100 (Tensor Cores & Throughput)
import torch
import time

def benchmark_a100():
    if not torch.cuda.is_available():
        print("CUDA não disponível para benchmark.")
        return

    print("Iniciando Teste de Throughput na A100 (Multiplicação de Matrizes 16384x16384 em FP16 / BF16)...")
    size = 16384
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    a = torch.randn(size, size, device='cuda', dtype=dtype)
    b = torch.randn(size, size, device='cuda', dtype=dtype)

    # Warmup
    torch.matmul(a[:1000, :1000], b[:1000, :1000])
    torch.cuda.synchronize()

    # Benchmark
    start = time.perf_counter()
    runs = 10
    for _ in range(runs):
        c = torch.matmul(a, b)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / runs

    # FLOPs = 2 * N^3
    tflops = (2 * (size ** 3)) / (elapsed * 1e12)
    print(f"\n✅ Benchmark Concluído!")
    print(f"⏱️ Tempo médio por iteração: {elapsed * 1000:.2f} ms")
    print(f"🚀 Performance Atingida: {tflops:.2f} TFLOPs ({dtype})")

benchmark_a100()

Iniciando Teste de Throughput na A100 (Multiplicação de Matrizes 16384x16384 em FP16 / BF16)...

✅ Benchmark Concluído!
⏱️ Tempo médio por iteração: 33.40 ms
🚀 Performance Atingida: 263.37 TFLOPs (torch.bfloat16)


In [ ]:
# 3. Instalação do Pacote Neural Áquila e Dependências de Visão/Embeddings
!pip install -q fastapi uvicorn pyngrok pydantic ultralytics face_recognition opencv-python-headless
print("✅ Dependências Neurais instaladas com sucesso!")

In [ ]:
# 4. Servidor de Offload Áquila Pro (FastAPI com Aceleração A100)
import nest_asyncio
from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel
from typing import List, Optional
import uvicorn
import threading

app = FastAPI(title="Áquila Pro A100 Neural Engine", version="2.1.0")

class TaskRequest(BaseModel):
    task_id: str
    task_type: str  # 'face_embeddings_batch', 'yolo_detect', 'tensor_crunch'
    payload: dict

@app.get("/status")
def get_status():
    gpu_info = {}
    if torch.cuda.is_available():
        gpu_info = {
            "device": torch.cuda.get_device_name(0),
            "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
            "vram_allocated_gb": round(torch.cuda.memory_allocated(0) / (1024**3), 2),
            "bf16_supported": torch.cuda.is_bf16_supported()
        }
    return {
        "engine": "Aquila_A100_Engine",
        "status": "READY",
        "gpu": gpu_info
    }

@app.post("/execute")
def execute_task(req: TaskRequest):
    start_t = time.perf_counter()
    # Exemplo de execução de carga pesada na A100
    if req.task_type == "tensor_crunch":
        dim = req.payload.get("dim", 8192)
        t1 = torch.randn(dim, dim, device="cuda", dtype=torch.float16)
        t2 = torch.randn(dim, dim, device="cuda", dtype=torch.float16)
        res = torch.matmul(t1, t2).sum().item()
        duration = time.perf_counter() - start_t
        return {"task_id": req.task_id, "status": "SUCCESS", "duration_ms": round(duration * 1000, 2), "result": res}

    return {"task_id": req.task_id, "status": "PROCESSED", "message": f"Task {req.task_type} concluída na GPU A100"}

print("✅ Servidor Neural Áquila pronto para aceitar conexões locais e remotas!")

In [ ]:
# 5. Iniciar Servidor e Expor Túnel Seguro (ngrok / Cloudflare)
# Se tiver um token ngrok gratuito, insira abaixo. Caso contrário, use localtunnel ou cloudflared.
import subprocess
import threading
import time

# Exemplo via cloudflared (sem necessidade de conta ou cadastro):
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

def run_app():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_app, daemon=True).start()
time.sleep(2)

print("🌐 Gerando URL Pública Segura via Cloudflare Tunnel...")
!cloudflared tunnel --url http://127.0.0.1:8000